# Image Classification Using Convolutional Neural Networks (CNNs)
## Project Overview
In this lightweight version of the project, we will build a minimal but high-quality Computer Vision image classifier using CNNs. We will use the **MNIST** dataset (handwritten digits), which is very small in file size and trains extremely quickly, making it perfect for an efficient, high-quality demonstration.

### Dataset
MNIST contains 60,000 28x28 grayscale images of handwritten digits (0 through 9).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from sklearn.metrics import confusion_matrix, classification_report

# Ensure matplotlib displays inline
%matplotlib inline

## 1. Dataset Loading & 2. Image Preprocessing
We load the MNIST dataset, normalize pixel values to [0, 1], and reshape the data to include the single grayscale channel.

In [ ]:
# Load MNIST dataset
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

# Reshape data to include the channel dimension (28, 28, 1)
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

class_names = [str(i) for i in range(10)]

print(f"Training data shape: {train_images.shape}")
print(f"Testing data shape: {test_images.shape}")

Let's visualize some sample digits from the training set:

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i].squeeze(), cmap='gray')
    plt.xlabel(class_names[train_labels[i]])
plt.show()

## 3. Model Building
We will construct a minimal CNN architecture. Because MNIST is a simpler dataset, a smaller network (fewer parameters) provides high quality while keeping the model size very lightweight.

In [ ]:
model = models.Sequential()

# First Convolutional Block (very lightweight)
model.add(layers.Conv2D(16, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))

# Second Convolutional Block
model.add(layers.Conv2D(32, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

# Dense Layers
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dropout(0.3)) # Prevent overfitting
model.add(layers.Dense(10, activation='softmax'))

model.summary()

## 4. Model Training
Compile the model and train it for just 5 epochs, which is plenty for MNIST to achieve high accuracy quickly.

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

epochs = 5
history = model.fit(train_images, train_labels, epochs=epochs, 
                    validation_data=(test_images, test_labels))

Visualize the training and validation accuracy and loss.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

plt.show()

## 5. Model Evaluation
Evaluate the test accuracy and generate a confusion matrix.

In [ ]:
test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)
print(f'Test accuracy: {test_acc:.4f}')

# Generate Predictions
predictions = model.predict(test_images)
predicted_labels = np.argmax(predictions, axis=1)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, predicted_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

print("\nClassification Report:\n", classification_report(test_labels, predicted_labels, target_names=class_names))

### Visualize Correct and Incorrect Predictions

In [ ]:
# Find indices of correct and incorrect predictions
correct_indices = np.where(predicted_labels == test_labels)[0]
incorrect_indices = np.where(predicted_labels != test_labels)[0]

def plot_images(indices, title):
    plt.figure(figsize=(10, 5))
    # Sometimes there are fewer than 10 errors, so we take the min
    num_images = min(10, len(indices))
    for i, idx in enumerate(indices[:num_images]):
        plt.subplot(2, 5, i + 1)
        plt.imshow(test_images[idx].squeeze(), cmap='gray')
        plt.title(f"T: {class_names[test_labels[idx]]}\nP: {class_names[predicted_labels[idx]]}", fontsize=10)
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_images(correct_indices, "Correctly Classified Digits")
if len(incorrect_indices) > 0:
    plot_images(incorrect_indices, "Incorrectly Classified Digits")